# 5. Componentes, series comerciales y selección del operacional
Filtro pasabajos Butterworth $N=3$, $W_n = 8900$ rad/s, implementado como cascada de una sección de primer orden ($R_3C_3$ + seguidor) y una celda Sallen-Key con $K=1$, $R_1=R_2=R$.

Este notebook reproduce los números y la figura de la Sección 5 del informe:
1. Ventana admisible de $W_n$ y verificación de la plantilla.
2. Cálculo de componentes y elección de valores comerciales.
3. Recalculo de la respuesta con valores comerciales.
4. Efecto de las tolerancias (peor caso y Monte Carlo).
5. Figura `respuesta_vs_plantilla.png`.
6. Verificación del TL082 (GBW, slew rate, saturación).
7. Tabla de valores esperados para el laboratorio.

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt
from pathlib import Path

FIG = Path('../Figuras'); FIG.mkdir(exist_ok=True)

fp, fs = 1000.0, 4000.0          # Hz
Ap, As = 1.0, 24.0               # dB
wp, ws = 2*np.pi*fp, 2*np.pi*fs
N = 3

## 1. Ventana admisible de $W_n$

In [ ]:
# A(w) = 10 log10(1 + (w/Wn)^(2N))
Wn_min = wp / (10**(Ap/10) - 1)**(1/(2*N))
Wn_max = ws / (10**(As/10) - 1)**(1/(2*N))
print(f'Wn_min = {Wn_min:.2f} rad/s  ({Wn_min/2/np.pi:.0f} Hz)')
print(f'Wn_max = {Wn_max:.2f} rad/s  ({Wn_max/2/np.pi:.0f} Hz)')

Wn = 8900.0
A_ideal = lambda w: 10*np.log10(1 + (w/Wn)**(2*N))
print(f'Con Wn = {Wn:.0f} rad/s: A(1 kHz) = {A_ideal(wp):.3f} dB, A(4 kHz) = {A_ideal(ws):.2f} dB')

## 2. Cálculo de componentes

In [ ]:
C3 = 10e-9
R3_ideal = 1/(Wn*C3)

C1, C2 = 4.7e-9, 18e-9           # Q = 0.5*sqrt(C2/C1) ~ 1  ->  C2 ~ 4*C1
R_ideal = 1/(Wn*np.sqrt(C1*C2))
print(f'R3 ideal = {R3_ideal/1e3:.2f} kOhm  -> comercial 11 kOhm (E24)')
print(f'R  ideal = {R_ideal/1e3:.2f} kOhm  -> comercial 12 kOhm (E12/E24)')
print(f'C2/C1 = {C2/C1:.3f}')

## 3. Respuesta con valores comerciales

In [ ]:
def H1(f, R3, C3):
    s = 2j*np.pi*np.asarray(f, dtype=float)
    return 1/(1 + s*R3*C3)

def H2(f, R1, R2, C1, C2):
    s = 2j*np.pi*np.asarray(f, dtype=float)
    return 1/(s**2*R1*R2*C1*C2 + s*C1*(R1+R2) + 1)

def Hc(f, R3, C3, R1, R2, C1, C2):
    return H1(f, R3, C3)*H2(f, R1, R2, C1, C2)

def Ht(f):
    s = 2j*np.pi*np.asarray(f, dtype=float)
    return Wn**3/((s + Wn)*(s**2 + Wn*s + Wn**2))

A = lambda H: -20*np.log10(np.abs(H))

nom = dict(R3=11e3, C3=10e-9, R1=12e3, R2=12e3, C1=4.7e-9, C2=18e-9)
w01 = 1/(nom['R3']*nom['C3'])
w02 = 1/np.sqrt(nom['R1']*nom['R2']*nom['C1']*nom['C2'])
Q = np.sqrt(nom['R1']*nom['R2']*nom['C1']*nom['C2'])/(nom['C1']*(nom['R1']+nom['R2']))
print(f'w01 = {w01:.0f} rad/s ({(w01/Wn-1)*100:+.2f} %)')
print(f'w02 = {w02:.0f} rad/s ({(w02/Wn-1)*100:+.2f} %)')
print(f'Q   = {Q:.3f} ({(Q-1)*100:+.2f} %)')
print(f'A(1 kHz) = {A(Hc(1e3, **nom)):.2f} dB   A(4 kHz) = {A(Hc(4e3, **nom)):.2f} dB')

f = np.logspace(2, 4.5, 200000)
g = -A(Hc(f, **nom))
print(f'f(-3 dB) = {f[np.argmin(abs(g+3))]:.1f} Hz')
print(f'max |Hc| = {np.abs(Hc(f, **nom)).max():.4f}   max |H2| aislada = {np.abs(H2(f, nom["R1"], nom["R2"], nom["C1"], nom["C2"])).max():.3f}')

## 4. Tolerancias: peor caso (64 extremos) y Monte Carlo (distribución uniforme)

In [ ]:
claves = ['R3', 'C3', 'R1', 'R2', 'C1', 'C2']
es_R = [True, False, True, True, False, False]

def peor_caso(tR, tC):
    a1, a4 = -np.inf, np.inf
    for sg in itertools.product((-1, 1), repeat=6):
        p = {k: nom[k]*(1 + s*(tR if r else tC)) for k, s, r in zip(claves, sg, es_R)}
        a1 = max(a1, A(Hc(1e3, **p))); a4 = min(a4, A(Hc(4e3, **p)))
    return a1, a4

def monte_carlo(tR, tC, n=20000, seed=1):
    rng = np.random.default_rng(seed)
    p = {k: nom[k]*(1 + rng.uniform(-1, 1, n)*(tR if r else tC)) for k, r in zip(claves, es_R)}
    a1, a4 = A(Hc(1e3, **p)), A(Hc(4e3, **p))
    return 100*np.mean((a1 <= Ap) & (a4 >= As))

print(' tol R | tol C | peor A(1k) | peor A(4k) | cumple (MC)')
for tR in (0.01, 0.05):
    for tC in (0.01, 0.02, 0.05, 0.10):
        a1, a4 = peor_caso(tR, tC)
        print(f' {tR*100:4.0f}% | {tC*100:4.0f}% | {a1:8.2f} dB | {a4:8.2f} dB | {monte_carlo(tR, tC):6.1f} %')

## 5. Figura: respuesta frente a la plantilla

In [ ]:
fg = np.logspace(2, np.log10(2e4), 1500)
rng = np.random.default_rng(7)
fig, ax = plt.subplots(figsize=(7.5, 4.5))
for _ in range(600):
    p = {k: nom[k]*(1 + rng.uniform(-0.05, 0.05)) for k in claves}
    ax.semilogx(fg, -A(Hc(fg, **p)), color='#9ec5ff', lw=0.4, alpha=0.25)
ax.semilogx(fg, -A(Ht(fg)), 'k--', lw=1.4, label='$H_t(s)$, $W_n$ = 8900 rad/s')
ax.semilogx(fg, -A(Hc(fg, **nom)), color='#2a6fdb', lw=2, label='$H_c(s)$, valores comerciales')
ax.plot([], [], color='#9ec5ff', lw=3, label='Dispersión R, C ±5 % (600 muestras)')
ax.fill_between([100, fp], -60, -Ap, color='#f2d7d5', lw=0)
ax.fill_between([fs, 2e4], -As, 5, color='#f2d7d5', lw=0, label='Zona prohibida')
ax.set_xlim(100, 2e4); ax.set_ylim(-60, 5)
ax.set_xlabel('Frecuencia [Hz]'); ax.set_ylabel('Ganancia [dB]')
ax.grid(True, which='both', color='#e3e3e3'); ax.legend(loc='lower left', fontsize=8.5)
fig.tight_layout(); fig.savefig(FIG/'respuesta_vs_plantilla.png', dpi=200)
plt.show()

## 6. Verificación del TL082 ($\pm 15$ V)

In [ ]:
GBW, SR_tip, SR_min, Vsal_min = 3e6, 13e6, 8e6, 12.0   # Hz, V/s, V/s, V (RL = 10 k)
f0 = w02/(2*np.pi); Vp = 2.0                # entrada de 4 Vpp

print(f'GBW / (Q f0) = {GBW/(Q*f0):.0f}')
print(f'Slew rate requerido a 1 kHz = {2*np.pi*1e3*Vp/1e6:.3f} V/us (mínimo TL082: {SR_min/1e6:.0f} V/us)')
print(f'Ancho de banda de potencia para 10 V pico = {SR_tip/(2*np.pi*10)/1e3:.0f} kHz (slew rate típico)')

# Efecto de la ganancia finita: seguidor con A(s) = GBW/s  ->  K(s) = A/(1+A)
def Hc_real(f, R3, C3, R1, R2, C1, C2):
    s = 2j*np.pi*np.asarray(f, dtype=float)
    K = (2*np.pi*GBW/s)/(1 + 2*np.pi*GBW/s)
    h1 = K/(1 + s*R3*C3)
    den = s**2*R1*R2*C1*C2 + s*(C1*(R1+R2) + C2*R1*(1-K)) + 1
    return h1*K/den
for fx in (1e3, 4e3):
    print(f'{fx/1e3:.0f} kHz: diferencia ideal vs GBW finito = {abs(A(Hc_real(fx, **nom)) - A(Hc(fx, **nom))):.1e} dB')
print(f'Margen de saturación con 2 V pico: {Vsal_min/Vp:.1f} veces')

## 7. Valores esperados para el laboratorio ($V_i$ = 4 Vpp, valores comerciales nominales)

In [ ]:
fl = [50,100,200,400,600,800,900,1000,1100,1250,1422,1600,1800,2000,2500,3000,3500,4000,4500,5000,6000,8000,10000,15000,20000]
fd = np.logspace(1, np.log10(3e4), 400000)
ph = np.degrees(np.unwrap(np.angle(Hc(fd, **nom))))
print('  f [Hz] |   A [dB] | fase [°] | Vo [Vpp] | dt [us]')
for fx in fl:
    h = Hc(fx, **nom); p = np.interp(fx, fd, ph)
    print(f'{fx:8d} | {-A(h):8.2f} | {p:8.1f} | {4*abs(h):8.4f} | {-p/360/fx*1e6:7.1f}')